<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## بازبینی مجدد پس از Final Health — بررسی متن اصلی Case 4 از 22

در بازبینی مجدد Human Review مشخص شد سؤال پذیرفته‌شده قبلی درباره «تمرین 8» مستقل و روشن نیست و Target Evidence فعلی نیز فقط شامل عنوان تمرین و یک دستور کوتاه است.

پیش از هر تصمیم REVISE یا RETARGET، باید متن واقعی همان کتاب در اطراف پاراگراف‌های 329 و 330 بررسی شود تا مشخص شود:

- تمرین 8 ادامه چه مبحثی بوده است؛
- هدف آموزشی آن از متن قبل یا بعد قابل استنباط هست یا نه؛
- آیا می‌توان از همان منبع سؤال طبیعی و مفهومی ساخت؛
- یا باید Target Evidence به بخش مناسب‌تری از همان موضوع تغییر کند.

این مرحله فقط Diagnostic است و هیچ Human Decision، Registry، Source، Citation یا Queue را تغییر نمی‌دهد.

</div>

In [1]:
# Code Cell 105

from pathlib import Path
import json
import re


CELL105_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL105_WORK_ID = (
    "book:trainer-training-financial-literacy"
)

CELL105_EDITION_ID = (
    "book:trainer-training-financial-literacy:fa-source"
)

CELL105_TARGET_PARAGRAPHS = {
    329,
    330,
}

CELL105_CONTEXT_START = 315
CELL105_CONTEXT_END = 345


cell105_failures = []


def cell105_check(name, condition):
    condition = bool(condition)

    if not condition:
        cell105_failures.append(
            name
        )

    return condition


def cell105_find_repo_root():
    candidates = []

    if "cell21_registry_path" in globals():
        registry_path = Path(
            cell21_registry_path
        ).resolve()

        candidates.extend(
            [
                registry_path.parent,
                *registry_path.parents,
            ]
        )

    cwd = Path.cwd().resolve()

    candidates.extend(
        [
            cwd,
            *cwd.parents,
        ]
    )

    seen = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in seen:
            continue

        seen.add(
            str(candidate)
        )

        if (
            (
                candidate
                /
                "data"
            ).exists()
            and
            (
                candidate
                /
                "notebooks"
            ).exists()
        ):
            return candidate

    return None


def cell105_extract_paragraph_number(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    direct = record.get(
        "paragraph_number"
    )

    if isinstance(
        direct,
        int,
    ):
        return direct

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "paragraph_number"
        )

        if isinstance(
            value,
            int,
        ):
            return value

    paragraph_id = (
        record.get(
            "paragraph_id"
        )
        or
        record.get(
            "document_id"
        )
    )

    if isinstance(
        paragraph_id,
        str,
    ):
        match = re.search(
            r":p0*(\d+)$",
            paragraph_id,
        )

        if match:
            return int(
                match.group(
                    1
                )
            )

    return None


def cell105_record_work_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "work_id"
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "work_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_edition_id(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = (
        record.get(
            "edition_id"
        )
        or
        record.get(
            "source_id"
        )
    )

    if isinstance(
        value,
        str,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "edition_id"
        )

        if isinstance(
            value,
            str,
        ):
            return value

    return None


def cell105_record_text(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    for key in [
        "text",
        "normalized_text",
        "content",
        "paragraph_text",
    ]:
        value = record.get(
            key
        )

        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):
            return value.strip()

    return None


def cell105_heading_path(record):
    if not isinstance(
        record,
        dict,
    ):
        return None

    value = record.get(
        "heading_path"
    )

    if isinstance(
        value,
        list,
    ):
        return value

    locator = record.get(
        "locator"
    )

    if isinstance(
        locator,
        dict,
    ):
        value = locator.get(
            "heading_path"
        )

        if isinstance(
            value,
            list,
        ):
            return value

    return None


def cell105_walk(obj):
    if isinstance(
        obj,
        dict,
    ):
        yield obj

        for value in obj.values():
            yield from cell105_walk(
                value
            )

    elif isinstance(
        obj,
        list,
    ):
        for item in obj:
            yield from cell105_walk(
                item
            )


cell105_repo_root = (
    cell105_find_repo_root()
)

cell105_check(
    "REPO_ROOT_NOT_FOUND",
    cell105_repo_root
    is not None,
)


cell105_registry_path = (
    (
        Path(
            cell21_registry_path
        ).resolve()
    )
    if "cell21_registry_path" in globals()
    else
    (
        cell105_repo_root
        /
        "data"
        /
        "phase2"
        /
        "human_review"
        /
        "human_review_registry.json"
    )
)


cell105_book_files = [
    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v1.json",

    cell105_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v2.json",
]


cell105_existing_book_files = [
    path
    for path in cell105_book_files
    if path.exists()
]


cell105_check(
    "BOOK_INGESTED_FILES_NOT_FOUND",
    bool(
        cell105_existing_book_files
    ),
)


cell105_registry = json.loads(
    cell105_registry_path.read_text(
        encoding="utf-8"
    )
)


cell105_case_matches = [
    case
    for case
    in cell105_registry.get(
        "completed_human_review_cases",
        [],
    )
    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL105_PILOT_ID
    )
]


cell105_check(
    "PILOT_CASE_NOT_UNIQUE",
    len(
        cell105_case_matches
    )
    ==
    1,
)


cell105_case = (
    cell105_case_matches[
        0
    ]
    if len(
        cell105_case_matches
    )
    ==
    1
    else
    {}
)


cell105_current_question = None

for draft in (
    cell105_case.get(
        "model_drafts",
        []
    )
    or
    []
):
    if (
        isinstance(
            draft,
            dict,
        )
        and
        draft.get(
            "human_decision"
        )
        ==
        "ACCEPT"
    ):
        cell105_current_question = (
            draft.get(
                "question_text"
            )
        )


cell105_current_answer = (
    (
        cell105_case.get(
            "expected_answer_review"
        )
        or
        {}
    ).get(
        "expected_answer_text"
    )
)


cell105_current_evidence = (
    cell105_case.get(
        "source_evidence",
        []
    )
    or
    []
)


cell105_book_records = []


for book_file in cell105_existing_book_files:
    payload = json.loads(
        book_file.read_text(
            encoding="utf-8"
        )
    )

    for record in cell105_walk(
        payload
    ):
        paragraph_number = (
            cell105_extract_paragraph_number(
                record
            )
        )

        text = cell105_record_text(
            record
        )

        if (
            paragraph_number
            is None
            or
            text
            is None
        ):
            continue

        work_id = cell105_record_work_id(
            record
        )

        edition_id = (
            cell105_record_edition_id(
                record
            )
        )

        paragraph_id = (
            record.get(
                "paragraph_id"
            )
            or
            record.get(
                "document_id"
            )
        )

        identity_text = " ".join(
            str(value)
            for value in [
                work_id,
                edition_id,
                paragraph_id,
            ]
            if value
        )

        exact_source_match = any([
            work_id
            ==
            CELL105_WORK_ID,

            edition_id
            ==
            CELL105_EDITION_ID,

            CELL105_EDITION_ID
            in
            identity_text,
        ])

        if not exact_source_match:
            continue

        if not (
            CELL105_CONTEXT_START
            <=
            paragraph_number
            <=
            CELL105_CONTEXT_END
        ):
            continue

        cell105_book_records.append({
            "paragraph_number":
                paragraph_number,

            "text":
                text,

            "heading_path":
                cell105_heading_path(
                    record
                ),

            "paragraph_id":
                paragraph_id,

            "source_file":
                str(
                    book_file
                ),
        })


cell105_unique_records = {}

for record in cell105_book_records:
    key = (
        record[
            "paragraph_number"
        ],
        record[
            "text"
        ],
    )

    if key not in cell105_unique_records:
        cell105_unique_records[
            key
        ] = record


cell105_context_records = sorted(
    cell105_unique_records.values(),
    key=lambda item: (
        item[
            "paragraph_number"
        ],
        item[
            "text"
        ],
    ),
)


cell105_found_numbers = {
    record[
        "paragraph_number"
    ]
    for record in cell105_context_records
}


cell105_check(
    "TARGET_PARAGRAPHS_NOT_FOUND_IN_BOOK",
    CELL105_TARGET_PARAGRAPHS.issubset(
        cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_BEFORE_TARGET",
    any(
        number
        <
        min(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


cell105_check(
    "NO_CONTEXT_AFTER_TARGET",
    any(
        number
        >
        max(
            CELL105_TARGET_PARAGRAPHS
        )
        for number
        in cell105_found_numbers
    ),
)


print(
    "=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ==="
)


print(
    "\n=== CURRENT ACCEPTED CASE ==="
)

print(
    "Pilot:",
    CELL105_PILOT_ID,
)

print(
    "Current Question:",
    cell105_current_question,
)

print(
    "Current Answer:",
    cell105_current_answer,
)

print(
    "Current Target Evidence:",
    cell105_case.get(
        "target_evidence_id"
    ),
)


print(
    "\n=== CURRENT PROTECTED SOURCE EVIDENCE ==="
)

for item in cell105_current_evidence:
    print(
        "\nDocument:",
        item.get(
            "document_id"
        ),
    )

    print(
        "Text:",
        item.get(
            "text"
        ),
    )


print(
    "\n=== EXACT BOOK CONTEXT ==="
)

print(
    "Work ID:",
    CELL105_WORK_ID,
)

print(
    "Edition ID:",
    CELL105_EDITION_ID,
)

print(
    "Requested paragraph window:",
    (
        CELL105_CONTEXT_START,
        CELL105_CONTEXT_END,
    ),
)

print(
    "Found unique records:",
    len(
        cell105_context_records
    ),
)


for record in cell105_context_records:
    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Paragraph:",
        record[
            "paragraph_number"
        ],
    )

    print(
        "Paragraph ID:",
        record[
            "paragraph_id"
        ],
    )

    print(
        "Heading Path:",
        record[
            "heading_path"
        ],
    )

    print(
        "Text:"
    )

    print(
        record[
            "text"
        ]
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    False,
)

print(
    "Human Decision added:",
    False,
)

print(
    "Question changed:",
    False,
)

print(
    "Answer changed:",
    False,
)

print(
    "Target changed:",
    False,
)

print(
    "Citation changed:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)

print(
    "Commit / Push:",
    False,
)


cell105_gate = (
    len(
        cell105_failures
    )
    ==
    0
)


print(
    "\n=== FAILURES ==="
)

print(
    cell105_failures
)


print(
    "\nCODE CELL 105 GATE:",
    (
        "PASS"
        if cell105_gate
        else
        "FAIL"
    ),
)


assert cell105_gate, (
    "Case 4 source-context diagnostic failed. "
    "Repair this SAME Code Cell 105 before any re-review decision."
)

=== CASE 4 POST-FINAL SOURCE CONTEXT DIAGNOSTIC ===

=== CURRENT ACCEPTED CASE ===
Pilot: question-pilot:909b695797935246c92c7a66ebe0f773
Current Question: در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟
Current Answer: باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.
Current Target Evidence: target-evidence:8d87341f7847558d90ee76e350c64fa3

=== CURRENT PROTECTED SOURCE EVIDENCE ===

Document: book:trainer-training-financial-literacy:fa-source:p000329
Text: تمرین 8

Document: book:trainer-training-financial-literacy:fa-source:p000330
Text: هر مقطع سنی دو نفر را انتخاب کنید و درباره میزان هزینه ماهانه آنها مصاحبه کنید. به آن‌ها اطمینان دهید که این اطلاعات محرمانه خواهد بود. نتیجه را تحلیل کنید.

=== EXACT BOOK CONTEXT ===
Work ID: book:trainer-training-financial-literacy
Edition ID: book:trainer-training-financial-literacy:fa-source


<div style="direction: rtl; text-align: right; white-space: normal; line-height: 1.8;">

## بازبینی دوم — مورد 4 از 22 — اصلاح کامل Target، سؤال و پاسخ

### 1. وضعیت ثبت‌شده در بازبینی اول

در Human Review اولیه، این Case با Target زیر پذیرفته شده بود:

**Target Evidence قبلی:**

`target-evidence:8d87341f7847558d90ee76e350c64fa3`

**متن منبع قبلی:**

**پاراگراف 329**

تمرین 8

**پاراگراف 330**

هر مقطع سنی دو نفر را انتخاب کنید و درباره میزان هزینه ماهانه آنها مصاحبه کنید. به آن‌ها اطمینان دهید که این اطلاعات محرمانه خواهد بود. نتیجه را تحلیل کنید.

**سؤال پذیرفته‌شده قبلی:**

در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟

**پاسخ پذیرفته‌شده قبلی:**

باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.

این سؤال و پاسخ در بازبینی اول ACCEPT شده بودند، اما در بازبینی دوم مشخص شد که:

- سؤال بدون مشاهده کتاب مستقل و طبیعی نیست؛
- عبارت «تمرین ۸» برای کاربر معنای مستقلی ندارد؛
- Target قبلی فقط عنوان و دستور یک تمرین را شامل می‌شود؛
- بخش اصلی و آموزشی موضوع بودجه‌بندی درست پیش از این تمرین قرار دارد؛
- بنابراین نگه‌داشتن این Case به شکل قبلی برای Golden Dataset مناسب نیست.

تصمیم بازبینی دوم:

**RETARGET**

نسخه قبلی پاک نمی‌شود و به‌عنوان تاریخچه بازبینی اول حفظ خواهد شد.

---

### 2. منبع بازبینی‌شده و جایگزین

در بررسی متن کامل کتاب، پاراگراف‌های 323 تا 327 به‌عنوان Target جایگزین انتخاب شدند.

#### پاراگراف 323

کودکان بین پنج تا هفت سال : اولین پولی که کودک دریافت می‌کند، تنها از آن نگهداری می‌کند و در واقع فقط پس‌انداز می‌کند و کم‌کم با موقعیت‌هایی که به او پول می‌دهیم (مانند فروشگاه یا مسافرت) خرج کردن را یاد می‌گیرد.

#### پاراگراف 324

کودکان بین هشت تا دوازده سال: دومین ردیف بودجه پول‌توجیبی هم یاد داده می‌شود. خرج پول با نظارت والدین است. هر چند به کودکان اجازهٔ اشتباه می‌دهند، اما همچنان کنارشان هستند و خرج پول با نظارت آن‌هاست. حتی اگر با خرید فرزند مخالف‌اند، اجازه می‌دهند بخرد. مهم این است که جلوی اشتباه گرفته نشود، بلکه بعد از اشتباه، خیلی واقعی درباره آن و چرایی اشتباه بودن آن گفتگو شود. در این سن، پول‌توجیبی فقط در دو ردیف خرج و پس‌انداز صرف می‌شود و می‌توان به هر کدام 50 درصد از دریافتی اختصاص یابد.

#### پاراگراف 325

اگر با این روش پیش رفته باشیم، دیگر «پول خودم است» مطرح نیست؛ چون پدر و مادر همواره با او هستند. هیچ وقت احساس نمی‌کند پول خودش است. اگر کودکان بگویند پول خودشان است،‌ می‌گوییم باید پاسخگوی تصمیم‌شان باشند. ممکن است تصمیم‌شان پیامدهایی داشته یا نداشته باشد. اگر پیامدها، فرصت آموزش را فراهم می‌کرد، تصمیم اشتباه و پیامدش را یادآوری می‌کنیم. یادمان باشد هنگام یادگیری راه رفتن، فرزند را برای زمین خوردن مواخذه نمی‌کنیم.

#### پاراگراف 326

نوجوان دوازده تا پانزده سال:در این دوره ردیف سرمایه‌گذاری هم اضافه می‌شود و پول‌توجیبی به سه بخش تقسیم می‌شود: خرج، پس‌انداز و سرمایه‌گذاری. در این حالت به ترتیب الگوی 50 درصد، 30 درصد و 20 درصد می‌تواند به کارگرفته شود.(تفاوت پس‌انداز و سرمایه‌گذاری: مشخص و نامشخص بودن هدف، بازه زمانی کوتاه مدت و بلندمدت). می‌توان در نشست خانوادگی دربارهٔ سرفصل‌های بودجه‌بندی و خرج با آن‌ها به توافق رسید. در این دوره می‌توان به شکل موقعیتی به او، هدیه به دیگران، بخشش یا حتی قرض به دیگران را پیشنهاد کرد.

#### پاراگراف 327

نوجوان پانزده تا هجده سال: در این دوره ردیف دیگری به پول‌توجیبی اضافه می‌شود: هدیه یا قرض به دیگران. تا پیش از این، به شکل موقعیتی با این ردیف آشنا بوده است. اما از اینجا به بعد با تعیین بخش و درصدی به این موضوع به آن ساختار می‌دهیم؛ الگوی 40 درصد خرج، 30 درصد پس‌انداز، 20 درصد سرمایه‌گذاری و 10 درصد هدیه به دیگران در سن مناسب است. نقش نظارتی والدین همچنان هست و آخر ماه از فرزندشان می‌خواهند توضیح دهد که این درصدها را چطور اعمال کرده است. این درصدها به نوعی خط‌کش است تا جلوی رفتارهای هیجانی نوجوان‌ها را بگیرد تا در هیچ کدام افراط و تفریط نداشته باشند؛ حتی در بخشش!

---

### 3. سؤال جایگزین

**مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟**

Human Reviewer این سؤال را در بازبینی دوم تأیید کرد.

---

### 4. پاسخ جایگزین

**در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.**

این پاسخ فقط از پاراگراف‌های 323 تا 327 همان منبع ساخته شده و دانش بیرونی به آن افزوده نشده است.

---

### 5. تصمیم نهایی بازبینی دوم

- تصمیم درباره Target قبلی: **RETARGET**
- Target قبلی: **حفظ در History**
- سؤال قبلی: **حفظ در History**
- پاسخ قبلی: **حفظ در History**
- Source خام: **بدون تغییر**
- Parser Output: **بدون تغییر**
- Target جدید: **پاراگراف‌های 323 تا 327**
- سؤال جدید: **ACCEPT**
- پاسخ جدید: **ACCEPT**
- Citation جدید: **مبتنی بر پاراگراف‌های 323 تا 327**
- Author Final Review: **NOT_STARTED**
- Case پس از این اصلاح همچنان آماده بازبینی دکتر کمیل رودی است.

</div>

In [2]:
# Code Cell 106
# ============================================================
# SECOND HUMAN REVIEW — CASE 4 / 22
#
# COMPLETE RETARGET + QUESTION + ANSWER + CITATION AMENDMENT
#
# Prerequisite:
#   Cell 105 diagnostic must have PASSed.
#
# This cell:
# - preserves the previously accepted target/question/answer/citation;
# - records the second-review RETARGET;
# - activates book paragraphs 323-327 as replacement evidence;
# - records the newly approved Question;
# - records the replacement Expected Answer;
# - rebuilds citation mapping from the replacement evidence;
# - updates BOTH Registry copies of the case;
# - keeps Author Final Review = NOT_STARTED;
# - writes a backup before changing Registry;
# - reloads and verifies the persisted Registry;
# - is idempotent for this exact amendment.
#
# It DOES NOT:
# - modify raw book source;
# - modify parser output;
# - delete first-review history;
# - record any Author decision;
# - commit or push anything.
# ============================================================

from pathlib import Path
from copy import deepcopy
from datetime import datetime, timezone
import hashlib
import json


# ============================================================
# 1) Fixed identity
# ============================================================

CELL106_PILOT_ID = (
    "question-pilot:"
    "909b695797935246c92c7a66ebe0f773"
)

CELL106_OLD_TARGET_ID = (
    "target-evidence:"
    "8d87341f7847558d90ee76e350c64fa3"
)

CELL106_REPLACEMENT_PARAGRAPHS = [
    323,
    324,
    325,
    326,
    327,
]

CELL106_NEW_QUESTION = (
    "مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند "
    "و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟"
)

CELL106_NEW_ANSWER = (
    "در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و "
    "پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی "
    "یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز "
    "تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. "
    "در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی "
    "۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری "
    "پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران "
    "نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، "
    "۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه "
    "به دیگران مطرح شده است."
)

CELL106_HUMAN_REASON = (
    "در بازبینی دوم مشخص شد Target قبلی فقط عنوان و دستور "
    "تمرین 8 را شامل می‌شد و سؤال پذیرفته‌شده قبلی بدون مشاهده "
    "کتاب، مستقل و طبیعی نبود. پس از بررسی متن کامل بخش "
    "بودجه‌بندی، پاراگراف‌های 323 تا 327 محتوای آموزشی کامل‌تر "
    "و مناسب‌تری برای همین Pilot فراهم کردند. Human Reviewer "
    "سؤال و پاسخ جایگزین را تأیید کرد."
)


# ============================================================
# 2) Utilities
# ============================================================

cell106_failures = []


def cell106_check(name, condition, details=None):

    if not bool(condition):

        failure = {
            "reason": name,
        }

        if details is not None:
            failure["details"] = details

        cell106_failures.append(
            failure
        )

    return bool(condition)


def cell106_sha256_text(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def cell106_stable_id(prefix, payload):

    if not isinstance(
        payload,
        str,
    ):
        payload = json.dumps(
            payload,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
        )

    digest = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:32]

    return (
        prefix
        +
        ":"
        +
        digest
    )


def cell106_now():

    return (
        datetime.now(
            timezone.utc
        )
        .replace(
            microsecond=0
        )
        .isoformat()
    )


# ============================================================
# 3) Cell 105 prerequisite
# ============================================================

cell106_check(
    "CELL105_GATE_NOT_PASS",
    globals().get(
        "cell105_gate"
    )
    is True,
)

cell106_check(
    "CELL105_REGISTRY_PATH_MISSING",
    "cell105_registry_path"
    in globals(),
)

cell106_check(
    "CELL105_CONTEXT_RECORDS_MISSING",
    "cell105_context_records"
    in globals(),
)

cell106_check(
    "CELL105_WORK_ID_MISSING",
    "CELL105_WORK_ID"
    in globals(),
)

cell106_check(
    "CELL105_EDITION_ID_MISSING",
    "CELL105_EDITION_ID"
    in globals(),
)


if cell106_failures:

    print(
        "Prerequisite failures:",
        cell106_failures,
    )

    raise AssertionError(
        "Cell 106 blocked because Cell 105 prerequisites are not valid."
    )


cell106_registry_path = Path(
    cell105_registry_path
).resolve()


cell106_check(
    "REGISTRY_FILE_NOT_FOUND",
    cell106_registry_path.exists(),
    str(
        cell106_registry_path
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 4) Resolve EXACT replacement paragraphs from Cell 105 output
# ============================================================

cell106_replacement_records = []


for paragraph_number in CELL106_REPLACEMENT_PARAGRAPHS:

    matches = [
        record

        for record
        in cell105_context_records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                "paragraph_number"
            )
            ==
            paragraph_number
        )
    ]

    cell106_check(
        (
            "REPLACEMENT_PARAGRAPH_"
            +
            str(
                paragraph_number
            )
            +
            "_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        cell106_replacement_records.append(
            deepcopy(
                matches[0]
            )
        )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement source resolution failed."
    )


cell106_check(
    "REPLACEMENT_ORDER_MISMATCH",
    [
        record[
            "paragraph_number"
        ]

        for record
        in cell106_replacement_records
    ]
    ==
    CELL106_REPLACEMENT_PARAGRAPHS,
)


cell106_replacement_text = "\n\n".join(
    record[
        "text"
    ]

    for record
    in cell106_replacement_records
)


# ============================================================
# 5) Source-grounding sanity checks
# ============================================================

CELL106_REQUIRED_SOURCE_CUES = [
    "پنج تا هفت",
    "هشت تا دوازده",
    "خرج و پس‌انداز",
    "50 درصد",
    "دوازده تا پانزده",
    "سرمایه‌گذاری",
    "30 درصد",
    "20 درصد",
    "پانزده تا هجده",
    "40 درصد",
    "10 درصد",
    "هدیه",
]


for cue in CELL106_REQUIRED_SOURCE_CUES:

    cell106_check(
        (
            "REQUIRED_SOURCE_CUE_MISSING:"
            +
            cue
        ),
        cue
        in
        cell106_replacement_text,
    )


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Replacement evidence does not support the approved Question/Answer."
    )


# ============================================================
# 6) Build normalized replacement source evidence
# ============================================================

cell106_source_evidence = []


for part_index, record in enumerate(
    cell106_replacement_records
):

    paragraph_number = (
        record[
            "paragraph_number"
        ]
    )

    document_id = (
        record[
            "paragraph_id"
        ]
    )

    text = (
        record[
            "text"
        ]
    )

    heading_path = (
        record.get(
            "heading_path"
        )
        or
        [
            "بودجه‌بندی"
        ]
    )

    cell106_source_evidence.append({
        "part_index":
            part_index,

        "role":
            "PROTECTED_REGION_MEMBER",

        "document_id":
            document_id,

        "text":
            text,

        "text_sha256":
            cell106_sha256_text(
                text
            ),

        "char_start":
            0,

        "char_end":
            len(
                text
            ),

        "locator": {
            "work_id":
                CELL105_WORK_ID,

            "edition_id":
                CELL105_EDITION_ID,

            "paragraph_id":
                document_id,

            "paragraph_number":
                paragraph_number,

            "heading_path":
                deepcopy(
                    heading_path
                ),
        },
    })


# ============================================================
# 7) Deterministic replacement IDs
# ============================================================

cell106_target_identity_payload = {
    "pilot_case_id":
        CELL106_PILOT_ID,

    "replacement_paragraphs":
        CELL106_REPLACEMENT_PARAGRAPHS,

    "replacement_documents": [
        {
            "document_id":
                item[
                    "document_id"
                ],

            "text_sha256":
                item[
                    "text_sha256"
                ],
        }

        for item
        in cell106_source_evidence
    ],
}


cell106_new_target_id = (
    cell106_stable_id(
        "target-evidence",
        cell106_target_identity_payload,
    )
)


cell106_new_question_id = (
    cell106_stable_id(
        "question-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question":
                CELL106_NEW_QUESTION,
        },
    )
)


cell106_new_answer_id = (
    cell106_stable_id(
        "expected-answer-draft",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "target_evidence_id":
                cell106_new_target_id,

            "answer":
                CELL106_NEW_ANSWER,
        },
    )
)


cell106_new_citation_id = (
    cell106_stable_id(
        "citation-mapping",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "documents": [
                item[
                    "document_id"
                ]

                for item
                in cell106_source_evidence
            ],
        },
    )
)


cell106_amendment_id = (
    cell106_stable_id(
        "second-human-review",
        {
            "pilot_case_id":
                CELL106_PILOT_ID,

            "old_target_evidence_id":
                CELL106_OLD_TARGET_ID,

            "new_target_evidence_id":
                cell106_new_target_id,

            "new_question_draft_id":
                cell106_new_question_id,

            "new_expected_answer_draft_id":
                cell106_new_answer_id,
        },
    )
)


# ============================================================
# 8) Load Registry fresh from disk
# ============================================================

cell106_registry_before_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_registry = json.loads(
    cell106_registry_before_text
)


cell106_registry_before_sha256 = (
    hashlib.sha256(
        cell106_registry_before_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 9) Locate Case in BOTH durable Registry copies
# ============================================================

cell106_completed_cases = (
    cell106_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell106_author_review_root = (
    cell106_registry.get(
        "author_review",
        {}
    )
    or
    {}
)


cell106_author_cases = (
    cell106_author_review_root.get(
        "cases_ready_for_review",
        [],
    )
)


def cell106_find_case(
    case_list,
    location_name,
):

    matches = [
        case

        for case
        in case_list

        if (
            isinstance(
                case,
                dict,
            )
            and
            case.get(
                "pilot_case_id"
            )
            ==
            CELL106_PILOT_ID
        )
    ]

    cell106_check(
        (
            location_name
            +
            "_CASE_NOT_UNIQUE"
        ),
        len(
            matches
        )
        ==
        1,
        {
            "match_count":
                len(
                    matches
                ),
        },
    )

    if len(
        matches
    ) == 1:

        return matches[0]

    return None


cell106_completed_case = (
    cell106_find_case(
        cell106_completed_cases,
        "COMPLETED",
    )
)


cell106_author_case = (
    cell106_find_case(
        cell106_author_cases,
        "AUTHOR_READY",
    )
)


if cell106_failures:

    print(
        cell106_failures
    )

    raise AssertionError(
        "Case 4 could not be resolved uniquely in Registry."
    )


# ============================================================
# 10) Helper: resolve currently active Question
# ============================================================

def cell106_active_question(case):

    final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    for draft in (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    ):

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            final_question_id
        ):

            return draft.get(
                "question_text"
            )

    question_review = (
        case.get(
            "question_review"
        )
        or
        {}
    )

    return question_review.get(
        "question_text"
    )


# ============================================================
# 11) Validate current / historical state before mutation
# ============================================================

CELL106_OLD_QUESTION = (
    "در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد "
    "چه مراحلی باید انجام شود؟"
)


for location_name, case in [
    (
        "COMPLETED",
        cell106_completed_case,
    ),
    (
        "AUTHOR_READY",
        cell106_author_case,
    ),
]:

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    existing_amendment_id = (
        existing_second_review.get(
            "amendment_id"
        )
    )

    exact_amendment_already_active = (
        existing_amendment_id
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    )

    if exact_amendment_already_active:

        continue

    cell106_check(
        (
            location_name
            +
            "_UNEXPECTED_EXISTING_SECOND_REVIEW"
        ),
        existing_amendment_id
        in
        [
            None,
            cell106_amendment_id,
        ],
        existing_amendment_id,
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_TARGET_MISMATCH"
        ),
        case.get(
            "target_evidence_id"
        )
        ==
        CELL106_OLD_TARGET_ID,
        case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        (
            location_name
            +
            "_OLD_QUESTION_MISMATCH"
        ),
        cell106_active_question(
            case
        )
        ==
        CELL106_OLD_QUESTION,
        cell106_active_question(
            case
        ),
    )

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    cell106_check(
        (
            location_name
            +
            "_AUTHOR_REVIEW_ALREADY_STARTED"
        ),
        author_final_review.get(
            "status"
        )
        ==
        "NOT_STARTED",
        author_final_review,
    )


if cell106_failures:

    print(
        "\n=== PRE-WRITE FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Registry current state does not match the expected Case 4 preconditions."
    )


# ============================================================
# 12) Reuse existing attribution — DO NOT invent attribution
# ============================================================

cell106_existing_citation_entries = (
    (
        cell106_completed_case.get(
            "citation_review"
        )
        or
        {}
    ).get(
        "citation_entries"
    )
    or
    []
)


cell106_attribution_template = None


for entry in cell106_existing_citation_entries:

    if (
        isinstance(
            entry,
            dict,
        )
        and
        isinstance(
            entry.get(
                "attribution"
            ),
            dict,
        )
    ):

        cell106_attribution_template = (
            deepcopy(
                entry[
                    "attribution"
                ]
            )
        )

        break


cell106_check(
    "EXISTING_ATTRIBUTION_TEMPLATE_NOT_FOUND",
    isinstance(
        cell106_attribution_template,
        dict,
    ),
)


if cell106_failures:

    raise AssertionError(
        cell106_failures
    )


# ============================================================
# 13) Build new Citation entries from exact replacement evidence
# ============================================================

cell106_new_citation_entries = []


for citation_index, item in enumerate(
    cell106_source_evidence
):

    cell106_new_citation_entries.append({
        "citation_index":
            citation_index,

        "document_id":
            item[
                "document_id"
            ],

        "source_id":
            CELL105_EDITION_ID,

        "target_evidence_id":
            cell106_new_target_id,

        "role":
            item[
                "role"
            ],

        "char_start":
            item[
                "char_start"
            ],

        "char_end":
            item[
                "char_end"
            ],

        "char_count":
            (
                item[
                    "char_end"
                ]
                -
                item[
                    "char_start"
                ]
            ),

        "locator":
            deepcopy(
                item[
                    "locator"
                ]
            ),

        "locator_origin":
            "COPIED_FROM_SECOND_HUMAN_REVIEW_REPLACEMENT_EVIDENCE",

        "attribution":
            deepcopy(
                cell106_attribution_template
            ),
    })


# ============================================================
# 14) Apply exact amendment to one Registry Case
# ============================================================

cell106_review_timestamp = (
    cell106_now()
)


def cell106_apply_amendment(
    case,
    registry_location,
):

    existing_second_review = (
        case.get(
            "second_human_review"
        )
        or
        {}
    )

    # --------------------------------------------------------
    # Idempotent no-op if this exact amendment is already active
    # --------------------------------------------------------

    if (
        existing_second_review.get(
            "amendment_id"
        )
        ==
        cell106_amendment_id
        and
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id
        and
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION
        and
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER
    ):

        return (
            "ALREADY_APPLIED"
        )


    # --------------------------------------------------------
    # Preserve exact first-review accepted state
    # --------------------------------------------------------

    previous_final_question_id = (
        case.get(
            "final_question_draft_id"
        )
    )

    previous_question_text = (
        cell106_active_question(
            case
        )
    )

    previous_expected_answer_review = (
        deepcopy(
            case.get(
                "expected_answer_review"
            )
        )
    )

    previous_citation_review = (
        deepcopy(
            case.get(
                "citation_review"
            )
        )
    )

    previous_source_evidence = (
        deepcopy(
            case.get(
                "source_evidence"
            )
        )
    )

    previous_target_id = (
        case.get(
            "target_evidence_id"
        )
    )


    history = case.setdefault(
        "post_final_human_review_history",
        [],
    )


    existing_history_ids = {
        item.get(
            "amendment_id"
        )

        for item
        in history

        if isinstance(
            item,
            dict,
        )
    }


    if (
        cell106_amendment_id
        not in
        existing_history_ids
    ):

        history.append({
            "amendment_id":
                cell106_amendment_id,

            "review_record_version":
                "second_human_review_amendment_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW",

            "review_timestamp_utc":
                cell106_review_timestamp,

            "registry_location":
                registry_location,

            "human_decision":
                "RETARGET_AND_ACCEPT_REPLACEMENT",

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "previous_state_preserved":
                True,

            "superseded_accepted_qa": {
                "target_evidence_id":
                    previous_target_id,

                "source_evidence":
                    previous_source_evidence,

                "question_draft_id":
                    previous_final_question_id,

                "question_text":
                    previous_question_text,

                "expected_answer_review":
                    previous_expected_answer_review,

                "citation_review":
                    previous_citation_review,
            },

            "replacement_state": {
                "target_evidence_id":
                    cell106_new_target_id,

                "replacement_paragraphs":
                    deepcopy(
                        CELL106_REPLACEMENT_PARAGRAPHS
                    ),

                "question_draft_id":
                    cell106_new_question_id,

                "question_text":
                    CELL106_NEW_QUESTION,

                "expected_answer_draft_id":
                    cell106_new_answer_id,

                "expected_answer_text":
                    CELL106_NEW_ANSWER,

                "citation_mapping_id":
                    cell106_new_citation_id,
            },

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,

            "author_review_recorded":
                False,
        })


    # --------------------------------------------------------
    # Mark old active draft as historically accepted but superseded
    # WITHOUT changing the old human_decision.
    # --------------------------------------------------------

    model_drafts = (
        case.get(
            "model_drafts",
            []
        )
        or
        []
    )


    for draft in model_drafts:

        if (
            isinstance(
                draft,
                dict,
            )
            and
            draft.get(
                "question_draft_id"
            )
            ==
            previous_final_question_id
        ):

            draft[
                "second_review_status"
            ] = (
                "SUPERSEDED_BY_RETARGET"
            )

            draft[
                "superseded_by_question_draft_id"
            ] = (
                cell106_new_question_id
            )


    existing_draft_ids = {
        draft.get(
            "question_draft_id"
        )

        for draft
        in model_drafts

        if isinstance(
            draft,
            dict,
        )
    }


    if (
        cell106_new_question_id
        not in
        existing_draft_ids
    ):

        previous_revisions = [
            int(
                draft.get(
                    "revision",
                    0,
                )
                or
                0
            )

            for draft
            in model_drafts

            if isinstance(
                draft,
                dict,
            )
        ]

        new_revision = (
            max(
                previous_revisions
                or
                [0]
            )
            +
            1
        )

        model_drafts.append({
            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "revision":
                new_revision,

            "machine_validation":
                "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

            "human_decision":
                "ACCEPT",

            "review_origin":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "target_evidence_id":
                cell106_new_target_id,
        })


    case[
        "model_drafts"
    ] = (
        model_drafts
    )


    # --------------------------------------------------------
    # Append second-review Human Review records
    # --------------------------------------------------------

    human_reviews = (
        case.get(
            "human_reviews",
            []
        )
        or
        []
    )


    cell106_retarget_review_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "RETARGET",
            },
        )
    )


    cell106_question_accept_event_id = (
        cell106_stable_id(
            "human-review",
            {
                "amendment_id":
                    cell106_amendment_id,

                "event":
                    "QUESTION_ACCEPT",
            },
        )
    )


    existing_review_event_ids = {
        review.get(
            "review_event_id"
        )

        for review
        in human_reviews

        if isinstance(
            review,
            dict,
        )
    }


    if (
        cell106_retarget_review_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_retarget_review_event_id,

            "review_record_version":
                "second_human_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_RETARGET",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "human_decision":
                "RETARGET",

            "previous_target_evidence_id":
                previous_target_id,

            "previous_question_draft_id":
                previous_final_question_id,

            "replacement_target_evidence_id":
                cell106_new_target_id,

            "replacement_question_draft_id":
                cell106_new_question_id,

            "previous_review_preserved":
                True,

            "human_review_reason":
                CELL106_HUMAN_REASON,

            "raw_source_mutated":
                False,

            "parser_output_mutated":
                False,
        })


    if (
        cell106_question_accept_event_id
        not in
        existing_review_event_ids
    ):

        human_reviews.append({
            "review_event_id":
                cell106_question_accept_event_id,

            "review_record_version":
                "second_human_question_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_QUESTION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "target_evidence_id":
                cell106_new_target_id,

            "question_draft_id":
                cell106_new_question_id,

            "question_text":
                CELL106_NEW_QUESTION,

            "human_decision":
                "ACCEPT",

            "human_note":
                (
                    "Human Reviewer explicitly approved the "
                    "replacement Question after reviewing the "
                    "full source context."
                ),

            "external_knowledge_used":
                False,
        })


    case[
        "human_reviews"
    ] = (
        human_reviews
    )


    # --------------------------------------------------------
    # Activate replacement Target + Question
    # --------------------------------------------------------

    case[
        "target_evidence_id"
    ] = (
        cell106_new_target_id
    )

    case[
        "source_evidence"
    ] = deepcopy(
        cell106_source_evidence
    )

    case[
        "final_question_draft_id"
    ] = (
        cell106_new_question_id
    )


    # --------------------------------------------------------
    # Activate replacement Expected Answer
    # --------------------------------------------------------

    case[
        "expected_answer_review"
    ] = {
        "expected_answer_draft_id":
            cell106_new_answer_id,

        "expected_answer_text":
            CELL106_NEW_ANSWER,

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "source_evidence":
            deepcopy(
                cell106_source_evidence
            ),

        "human_review": {
            "review_record_version":
                "second_human_expected_answer_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_EXPECTED_ANSWER",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "human_decision":
                "ACCEPT",

            "expected_answer_text":
                CELL106_NEW_ANSWER,

            "external_knowledge_used":
                False,

            "human_context_used_as_evidence":
                False,

            "human_note":
                (
                    "Human Reviewer accepted the replacement "
                    "Expected Answer after reviewing the exact "
                    "book paragraphs 323-327."
                ),
        },
    }


    # --------------------------------------------------------
    # Activate replacement Citation Mapping
    # --------------------------------------------------------

    case[
        "citation_review"
    ] = {
        "citation_mapping_id":
            cell106_new_citation_id,

        "citation_entries":
            deepcopy(
                cell106_new_citation_entries
            ),

        "human_decision":
            "ACCEPT",

        "machine_validation_status":
            "SECOND_HUMAN_REVIEW_LOCAL_GATES_PASS",

        "human_review": {
            "review_record_version":
                "second_human_citation_review_record_v1",

            "review_stage":
                "SECOND_HUMAN_REVIEW_CITATION",

            "pilot_case_id":
                CELL106_PILOT_ID,

            "question_draft_id":
                cell106_new_question_id,

            "expected_answer_draft_id":
                cell106_new_answer_id,

            "target_evidence_id":
                cell106_new_target_id,

            "citation_mapping_id":
                cell106_new_citation_id,

            "human_decision":
                "ACCEPT",

            "external_locator_lookup_used":
                False,

            "human_note":
                (
                    "Citation Mapping uses only the exact "
                    "replacement book paragraphs 323-327 "
                    "resolved in Cell 105."
                ),
        },
    }


    # --------------------------------------------------------
    # Explicit second-review current status
    # --------------------------------------------------------

    case[
        "second_human_review"
    ] = {
        "amendment_id":
            cell106_amendment_id,

        "status":
            "COMPLETE",

        "decision":
            "RETARGET_AND_ACCEPT_REPLACEMENT",

        "review_timestamp_utc":
            cell106_review_timestamp,

        "previous_target_evidence_id":
            previous_target_id,

        "replacement_target_evidence_id":
            cell106_new_target_id,

        "replacement_question_draft_id":
            cell106_new_question_id,

        "replacement_expected_answer_draft_id":
            cell106_new_answer_id,

        "replacement_citation_mapping_id":
            cell106_new_citation_id,

        "previous_state_preserved":
            True,

        "raw_source_mutated":
            False,

        "parser_output_mutated":
            False,

        "author_review_status":
            "NOT_STARTED",
    }


    # --------------------------------------------------------
    # Keep author review untouched / pending
    # --------------------------------------------------------

    author_final_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )

    author_final_review.setdefault(
        "reviewer",
        "AUTHOR",
    )

    author_final_review[
        "status"
    ] = (
        "NOT_STARTED"
    )

    author_final_review[
        "decision"
    ] = None

    author_final_review[
        "notes"
    ] = None

    case[
        "author_final_review"
    ] = (
        author_final_review
    )


    case[
        "final_status"
    ] = (
        "HUMAN_REVIEW_COMPLETE_"
        "AUTHOR_FINAL_REVIEW_PENDING"
    )


    return (
        "APPLIED"
    )


# ============================================================
# 15) Apply to BOTH durable copies
# ============================================================

cell106_completed_apply_status = (
    cell106_apply_amendment(
        cell106_completed_case,
        "completed_human_review_cases",
    )
)


cell106_author_apply_status = (
    cell106_apply_amendment(
        cell106_author_case,
        "author_review.cases_ready_for_review",
    )
)


# ============================================================
# 16) In-memory validation BEFORE write
# ============================================================

def cell106_validate_final_case(
    case,
    location_name,
):

    failures_before = len(
        cell106_failures
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_TARGET_MISMATCH",
        case.get(
            "target_evidence_id"
        )
        ==
        cell106_new_target_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_ID_MISMATCH",
        case.get(
            "final_question_draft_id"
        )
        ==
        cell106_new_question_id,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_QUESTION_TEXT_MISMATCH",
        cell106_active_question(
            case
        )
        ==
        CELL106_NEW_QUESTION,
    )

    cell106_check(
        location_name
        +
        "_ACTIVE_ANSWER_MISMATCH",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        CELL106_NEW_ANSWER,
    )

    cell106_check(
        location_name
        +
        "_ANSWER_NOT_ACCEPTED",
        (
            case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_NOT_ACCEPTED",
        (
            case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "human_decision"
        )
        ==
        "ACCEPT",
    )

    cell106_check(
        location_name
        +
        "_CITATION_TARGET_MISMATCH",
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            cell106_new_target_id

            for entry
            in (
                (
                    case.get(
                        "citation_review"
                    )
                    or
                    {}
                ).get(
                    "citation_entries"
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_REPLACEMENT_EVIDENCE_COUNT_MISMATCH",
        len(
            case.get(
                "source_evidence",
                []
            )
            or
            []
        )
        ==
        5,
    )

    cell106_check(
        location_name
        +
        "_SECOND_REVIEW_NOT_COMPLETE",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "COMPLETE",
    )

    cell106_check(
        location_name
        +
        "_AMENDMENT_ID_MISMATCH",
        (
            case.get(
                "second_human_review"
            )
            or
            {}
        ).get(
            "amendment_id"
        )
        ==
        cell106_amendment_id,
    )

    cell106_check(
        location_name
        +
        "_HISTORY_MISSING",
        any(
            (
                isinstance(
                    record,
                    dict,
                )
                and
                record.get(
                    "amendment_id"
                )
                ==
                cell106_amendment_id
            )

            for record
            in (
                case.get(
                    "post_final_human_review_history",
                    []
                )
                or
                []
            )
        ),
    )

    cell106_check(
        location_name
        +
        "_AUTHOR_REVIEW_STATUS_CHANGED",
        (
            case.get(
                "author_final_review"
            )
            or
            {}
        ).get(
            "status"
        )
        ==
        "NOT_STARTED",
    )

    return (
        len(
            cell106_failures
        )
        ==
        failures_before
    )


cell106_validate_final_case(
    cell106_completed_case,
    "COMPLETED",
)


cell106_validate_final_case(
    cell106_author_case,
    "AUTHOR_READY",
)


cell106_check(
    "AUTHOR_REVIEW_CASE_COUNT_CHANGED",
    cell106_author_review_root.get(
        "case_count"
    )
    ==
    22,
    cell106_author_review_root.get(
        "case_count"
    ),
)


if cell106_failures:

    print(
        "\n=== IN-MEMORY VALIDATION FAILURES ==="
    )

    print(
        cell106_failures
    )

    raise AssertionError(
        "Second Review amendment failed before Registry write."
    )


# ============================================================
# 17) Write only if bytes actually need to change
# ============================================================

cell106_registry_after_text = (
    json.dumps(
        cell106_registry,
        ensure_ascii=False,
        indent=2,
    )
    +
    "\n"
)


cell106_change_required = (
    cell106_registry_after_text
    !=
    cell106_registry_before_text
)


cell106_backup_path = None


if cell106_change_required:

    timestamp_for_filename = (
        datetime.now(
            timezone.utc
        ).strftime(
            "%Y%m%dT%H%M%SZ"
        )
    )

    cell106_backup_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.stem
            +
            ".before_case4_second_review_"
            +
            timestamp_for_filename
            +
            cell106_registry_path.suffix
        )
    )

    cell106_backup_path.write_text(
        cell106_registry_before_text,
        encoding="utf-8",
    )


    cell106_temp_path = (
        cell106_registry_path.parent
        /
        (
            cell106_registry_path.name
            +
            ".cell106.tmp"
        )
    )


    cell106_temp_path.write_text(
        cell106_registry_after_text,
        encoding="utf-8",
    )


    # Validate temporary JSON before replacing Registry.
    json.loads(
        cell106_temp_path.read_text(
            encoding="utf-8"
        )
    )


    cell106_temp_path.replace(
        cell106_registry_path
    )


# ============================================================
# 18) Reload exact persisted Registry
# ============================================================

cell106_persisted_text = (
    cell106_registry_path.read_text(
        encoding="utf-8"
    )
)


cell106_persisted_registry = (
    json.loads(
        cell106_persisted_text
    )
)


cell106_registry_after_sha256 = (
    hashlib.sha256(
        cell106_persisted_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 19) Resolve persisted copies independently
# ============================================================

cell106_persisted_completed_matches = [
    case

    for case
    in cell106_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_persisted_author_matches = [
    case

    for case
    in (
        (
            cell106_persisted_registry.get(
                "author_review",
                {}
            )
            or
            {}
        ).get(
            "cases_ready_for_review",
            [],
        )
    )

    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL106_PILOT_ID
    )
]


cell106_check(
    "PERSISTED_COMPLETED_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_completed_matches
    )
    ==
    1,
)


cell106_check(
    "PERSISTED_AUTHOR_CASE_NOT_UNIQUE",
    len(
        cell106_persisted_author_matches
    )
    ==
    1,
)


if not cell106_failures:

    cell106_persisted_completed_case = (
        cell106_persisted_completed_matches[
            0
        ]
    )

    cell106_persisted_author_case = (
        cell106_persisted_author_matches[
            0
        ]
    )

    cell106_validate_final_case(
        cell106_persisted_completed_case,
        "PERSISTED_COMPLETED",
    )

    cell106_validate_final_case(
        cell106_persisted_author_case,
        "PERSISTED_AUTHOR_READY",
    )


# ============================================================
# 20) Cross-copy canonical equality
# ============================================================

if not cell106_failures:

    cell106_check(
        "CROSS_COPY_TARGET_MISMATCH",
        cell106_persisted_completed_case.get(
            "target_evidence_id"
        )
        ==
        cell106_persisted_author_case.get(
            "target_evidence_id"
        ),
    )

    cell106_check(
        "CROSS_COPY_QUESTION_MISMATCH",
        cell106_active_question(
            cell106_persisted_completed_case
        )
        ==
        cell106_active_question(
            cell106_persisted_author_case
        ),
    )

    cell106_check(
        "CROSS_COPY_ANSWER_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "expected_answer_review"
            )
            or
            {}
        ).get(
            "expected_answer_text"
        ),
    )

    cell106_check(
        "CROSS_COPY_CITATION_ID_MISMATCH",
        (
            cell106_persisted_completed_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        )
        ==
        (
            cell106_persisted_author_case.get(
                "citation_review"
            )
            or
            {}
        ).get(
            "citation_mapping_id"
        ),
    )


# ============================================================
# 21) Final gate
# ============================================================

cell106_gate = (
    len(
        cell106_failures
    )
    ==
    0
)


print(
    "=== CASE 4 SECOND HUMAN REVIEW — COMPLETE RETARGET ==="
)


print(
    "\nPilot:",
    CELL106_PILOT_ID,
)


print(
    "\nPrevious Target:",
    CELL106_OLD_TARGET_ID,
)


print(
    "Replacement Target:",
    cell106_new_target_id,
)


print(
    "Replacement paragraphs:",
    CELL106_REPLACEMENT_PARAGRAPHS,
)


print(
    "\nReplacement Question:"
)

print(
    CELL106_NEW_QUESTION
)


print(
    "\nReplacement Answer:"
)

print(
    CELL106_NEW_ANSWER
)


print(
    "\nQuestion Draft ID:",
    cell106_new_question_id,
)


print(
    "Expected Answer Draft ID:",
    cell106_new_answer_id,
)


print(
    "Citation Mapping ID:",
    cell106_new_citation_id,
)


print(
    "Second Review Amendment ID:",
    cell106_amendment_id,
)


print(
    "\nCompleted copy apply status:",
    cell106_completed_apply_status,
)


print(
    "Author-ready copy apply status:",
    cell106_author_apply_status,
)


print(
    "\nRegistry write required:",
    cell106_change_required,
)


print(
    "Backup path:",
    (
        str(
            cell106_backup_path
        )
        if cell106_backup_path
        is not None
        else
        "NOT_NEEDED_ALREADY_APPLIED"
    ),
)


print(
    "\nRegistry SHA256 before:",
    cell106_registry_before_sha256,
)


print(
    "Registry SHA256 after:",
    cell106_registry_after_sha256,
)


print(
    "\n=== HISTORY / SAFETY ==="
)


print(
    "Previous Target preserved:",
    True,
)


print(
    "Previous Question preserved:",
    True,
)


print(
    "Previous Answer preserved:",
    True,
)


print(
    "Previous Citation preserved:",
    True,
)


print(
    "Raw Source mutated:",
    False,
)


print(
    "Parser Output mutated:",
    False,
)


print(
    "Author Review recorded:",
    False,
)


print(
    "Author Review status:",
    "NOT_STARTED",
)


print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)


print(
    cell106_failures
)


print(
    "\nCODE CELL 106 GATE:",
    (
        "PASS"
        if cell106_gate
        else
        "FAIL"
    ),
)


assert cell106_gate, (
    "Case 4 Second Human Review amendment failed. "
    "Do not continue to another Case until this SAME Cell 106 is repaired."
)

=== CASE 4 SECOND HUMAN REVIEW — COMPLETE RETARGET ===

Pilot: question-pilot:909b695797935246c92c7a66ebe0f773

Previous Target: target-evidence:8d87341f7847558d90ee76e350c64fa3
Replacement Target: target-evidence:63c2f0f041fa8caa0f21a1aed165d7ec
Replacement paragraphs: [323, 324, 325, 326, 327]

Replacement Question:
مدیریت پول‌توجیبی از ۵ تا ۱۸ سالگی چگونه تغییر می‌کند و در هر بازه سنی چه تقسیم‌بندی‌ای پیشنهاد شده است؟

Replacement Answer:
در ۵ تا ۷ سالگی کودک ابتدا بیشتر پول را نگه می‌دارد و پس‌انداز می‌کند و به‌تدریج خرج‌کردن را در موقعیت‌های واقعی یاد می‌گیرد. در ۸ تا ۱۲ سالگی پول‌توجیبی به خرج و پس‌انداز تقسیم می‌شود و الگوی پیشنهادی ۵۰ درصد برای هر کدام است. در ۱۲ تا ۱۵ سالگی سرمایه‌گذاری هم اضافه می‌شود و الگوی ۵۰ درصد خرج، ۳۰ درصد پس‌انداز و ۲۰ درصد سرمایه‌گذاری پیشنهاد می‌شود. در ۱۵ تا ۱۸ سالگی هدیه یا قرض به دیگران نیز به بودجه اضافه می‌شود و الگوی ۴۰ درصد خرج، ۳۰ درصد پس‌انداز، ۲۰ درصد سرمایه‌گذاری و ۱۰ درصد هدیه به دیگران مطرح شده است.

Question Draft ID: question-draft:a0